Interpretação dos LRi e importância das componentes RF

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append("/code/scripts")

from analysis_config import (
    ANALYSIS,
    FEATURE_LABELS,
    LULC_CLASSES,
    LULC_MODELLED_CLASSES,
    RESULTS,
    lulc_periods,
    model_interpretation_configurations
)
from model_analysis_utils import (
    lri_by_class,
    lri_lulc_by_period,
    raster_valid_cell_count,
    raster_value_frequency,
    rf_feature_importance,
    rf_model_configuration
)

In [ ]:
configurations = model_interpretation_configurations()

out_dir = ANALYSIS / "01_model_interpretation"
out_dir.mkdir(parents=True, exist_ok=True)
output = out_dir / "model_interpretation.xlsx"

lulc_nomenclature = pd.DataFrame([
    {
        "class_code": code,
        "class_label": LULC_CLASSES[code],
        "included_in_model": code in LULC_MODELLED_CLASSES
    }
    for code in sorted(LULC_CLASSES)
])

In [ ]:
lri_tables = []
lulc_period_tables = []
lulc_final_tables = []
importance_detail = []
importance_summary = []
valid_cells_by_area = {}

for config in configurations:
    area = config["area"]
    scenario_lr = config["scenario_lr"]
    scenario_rf = config["scenario_rf"]
    source = config["source"]

    processed = Path(f"/code/data/processed/{area}")
    final_lr = RESULTS / area / scenario_lr / "final"
    reference = (
        processed / "topo" / "reclassified" /
        f"rcls_dem_{area}.tif"
    )

    if area not in valid_cells_by_area:
        valid_cells_by_area[area] = raster_valid_cell_count(reference)

    variables = {
        "dem": (
            processed / "topo" / "reclassified" /
            f"rcls_dem_{area}.tif",
            final_lr / "lri_dem.tif"
        ),
        "slope": (
            processed / "topo" / "reclassified" /
            f"rcls_slope_{area}.tif",
            final_lr / "lri_slope.tif"
        )
    }

    for variable, paths in variables.items():
        lri_tables.append(
            lri_by_class(
                class_raster=paths[0],
                lri_raster=paths[1],
                variable=variable,
                area=area,
                scenario=scenario_lr,
                source=source
            )
        )

    for period in lulc_periods(area, scenario_lr):
        lulc_period_tables.append(
            lri_lulc_by_period(
                lulc_raster=period["lulc_raster"],
                burned_raster=period["burned_raster"],
                reference_raster=reference,
                lulc_year=period["lulc_year"],
                burned_period=period["burned_period"],
                weight=period["weight"],
                total_cells=valid_cells_by_area[area],
                area=area,
                scenario=scenario_lr,
                source=source
            )
        )

    lulc_final_tables.append(
        raster_value_frequency(
            final_lr / "lri_lulc.tif",
            value_name="lri",
            area=area,
            scenario=scenario_lr,
            source=source,
            variable="lulc_final_weighted"
        )
    )

    models_xlsx = (
        RESULTS / area / scenario_rf /
        "lri_model" / "models.xlsx"
    )
    models, features, _ = rf_model_configuration(models_xlsx)
    detail, summary = rf_feature_importance(
        models,
        features,
        area=area,
        scenario=scenario_rf
    )

    importance_detail.append(detail)
    importance_summary.append(summary)

In [ ]:
lri_topography = pd.concat(lri_tables, ignore_index=True)
lri_lulc_period = pd.concat(lulc_period_tables, ignore_index=True)
lri_lulc_final = pd.concat(lulc_final_tables, ignore_index=True)
rf_detail = pd.concat(importance_detail, ignore_index=True)
rf_summary = pd.concat(importance_summary, ignore_index=True)

with pd.ExcelWriter(output) as writer:
    lri_topography.to_excel(
        writer,
        sheet_name="LRi_topography",
        index=False
    )
    lri_lulc_period.to_excel(
        writer,
        sheet_name="LRi_LULC_by_period",
        index=False
    )
    lri_lulc_final.to_excel(
        writer,
        sheet_name="LRi_LULC_final_dist",
        index=False
    )
    lulc_nomenclature.to_excel(
        writer,
        sheet_name="LULC_nomenclature",
        index=False
    )
    rf_summary.to_excel(
        writer,
        sheet_name="RF_importance_summary",
        index=False
    )
    rf_detail.to_excel(
        writer,
        sheet_name="RF_importance_replicates",
        index=False
    )

print("Resultados guardados em:", output)
rf_summary